# Gold Layer - Analytical Data Modeling

Gold contains purpose-specific analytical datasets. Every one-to-many table is aggregated to the target grain before it is joined.


## Objective

Build customer-, order-, product-, seller-, delivery-, and ML-ready datasets without accidental row multiplication.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Olist-Gold-Analytics")
    .getOrCreate()
)
SILVER_PATH = "../../data/silver"
GOLD_PATH = "../../data/gold"
TABLES = ["customers", "orders", "order_items", "order_payments", "order_reviews", "products", "sellers", "geolocation"]

## Load Silver Data

In [ ]:
missing_tables = [table for table in TABLES if not os.path.exists(os.path.join(SILVER_PATH, table))]
if missing_tables:
    raise FileNotFoundError(
        f"Missing Silver tables: {missing_tables}. Run 02_silver_transformation.ipynb first."
    )

silver = {table: spark.read.parquet(os.path.join(SILVER_PATH, table)) for table in TABLES}
silver_counts = {table: frame.count() for table, frame in silver.items()}
silver_counts

## Define Table Grains

| Dataset | Target grain | Key validation |
|---|---|---|
| customer_analytics | One row per `customer_unique_id` | No duplicate customer keys |
| sales_analytics | One row per `order_id` | No duplicate order keys |
| product_analytics | One row per `product_id` | No duplicate product keys |
| seller_analytics | One row per `seller_id` | No duplicate seller keys |
| delivery_analytics | One row per `order_id` | No duplicate order keys |
| ml_customer_repeat | One row per `customer_unique_id` | Target is based only on historical order count |

In [ ]:
customers = silver["customers"]
orders = silver["orders"]
order_items = silver["order_items"]
payments = silver["order_payments"]
reviews = silver["order_reviews"]
products = silver["products"]
sellers = silver["sellers"]

## Customer Analytics

First aggregate each one-to-many source to order grain. Only then roll up to `customer_unique_id`.

In [ ]:
items_by_order = (
    order_items.groupBy("order_id")
    .agg(
        F.count("*").alias("item_count"),
        F.sum("price").alias("item_value"),
        F.sum("freight_value").alias("freight_value"),
        F.countDistinct("product_id").alias("unique_products"),
        F.countDistinct("seller_id").alias("unique_sellers"),
    )
)
payments_by_order = payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("payment_value"),
    F.count("*").alias("payment_record_count"),
    F.max("payment_installments").alias("max_payment_installments"),
)
reviews_by_order = reviews.groupBy("order_id").agg(
    F.avg("review_score").alias("average_review_score"),
    F.count("review_score").alias("review_count"),
)

order_facts = (
    orders.join(customers.select("customer_id", "customer_unique_id", "customer_city", "customer_state"), "customer_id", "left")
    .join(items_by_order, "order_id", "left")
    .join(payments_by_order, "order_id", "left")
    .join(reviews_by_order, "order_id", "left")
)

customer_analytics = (
    order_facts.groupBy("customer_unique_id")
    .agg(
        F.first("customer_city", ignorenulls=True).alias("customer_city"),
        F.first("customer_state", ignorenulls=True).alias("customer_state"),
        F.countDistinct("order_id").alias("total_orders"),
        F.sum(F.coalesce("item_count", F.lit(0))).alias("total_items"),
        F.sum(F.coalesce("item_value", F.lit(0.0))).alias("total_item_value"),
        F.sum(F.coalesce("freight_value", F.lit(0.0))).alias("total_freight_value"),
        F.sum(F.coalesce("payment_value", F.lit(0.0))).alias("total_payment_value"),
        F.min("order_purchase_timestamp").alias("first_order_date"),
        F.max("order_purchase_timestamp").alias("last_order_date"),
        F.avg("average_review_score").alias("average_review_score"),
    )
    .withColumn("repeat_customer", (F.col("total_orders") > 1).cast("int"))
)

## Sales Analytics

Definition: `total_order_value` is item value plus freight value. Payment value is kept separately because it represents payment records and should not be blindly added to item revenue.

In [ ]:
sales_analytics = (
    order_facts.select(
        "order_id", "customer_unique_id", "customer_state",
        "order_purchase_timestamp", "order_status",
        "item_count", "item_value", "freight_value",
        "payment_value", "payment_record_count",
        "unique_products", "unique_sellers", "average_review_score",
    )
    .withColumn("total_order_value", F.coalesce(F.col("item_value"), F.lit(0.0)) + F.coalesce(F.col("freight_value"), F.lit(0.0)))
)

## Product Analytics

In [ ]:
product_analytics = (
    order_items.join(products.select("product_id", "product_category_name"), "product_id", "left")
    .groupBy("product_id", "product_category_name")
    .agg(
        F.count("*").alias("total_units_sold"),
        F.sum("price").alias("total_sales_value"),
        F.avg("price").alias("average_selling_price"),
        F.avg("freight_value").alias("average_freight_value"),
        F.countDistinct("order_id").alias("unique_orders"),
        F.countDistinct("seller_id").alias("unique_sellers"),
    )
)

## Seller Analytics

In [ ]:
seller_analytics = (
    order_items.join(sellers.select("seller_id", "seller_city", "seller_state"), "seller_id", "left")
    .groupBy("seller_id", "seller_city", "seller_state")
    .agg(
        F.count("*").alias("total_items_sold"),
        F.countDistinct("order_id").alias("total_orders_served"),
        F.sum("price").alias("total_sales_value"),
        F.avg("price").alias("average_item_price"),
        F.avg("freight_value").alias("average_freight_value"),
        F.countDistinct("product_id").alias("unique_products"),
    )
)

## Delivery Analytics

`delivery_delay_days` is actual delivery date minus estimated delivery date. Negative means early; positive means late. Missing delivery timestamps remain null.

In [ ]:
delivery_analytics = (
    order_facts.select(
        "order_id", "customer_unique_id", "customer_state",
        "order_purchase_timestamp", "order_delivered_carrier_date",
        "order_delivered_customer_date", "order_estimated_delivery_date",
        "order_status", "average_review_score",
    )
    .withColumn("actual_delivery_days", F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))
    .withColumn("estimated_delivery_days", F.datediff("order_estimated_delivery_date", "order_purchase_timestamp"))
    .withColumn("delivery_delay_days", F.datediff("order_delivered_customer_date", "order_estimated_delivery_date"))
    .withColumn("delivery_status", F.when(F.col("delivery_delay_days").isNull(), F.lit(None).cast("string")).when(F.col("delivery_delay_days") < 0, F.lit("Early")).when(F.col("delivery_delay_days") > 0, F.lit("Late")).otherwise(F.lit("On Time")))
)

## ML-Ready Dataset

A realistic first ML problem is repeat-purchase prediction. This notebook creates a descriptive customer-level target, but does not train a model. A production training split would need a cutoff date so features precede the target window.

In [ ]:
ml_customer_repeat = customer_analytics.select(
    "customer_unique_id", "customer_state", "total_orders",
    "total_items", "total_item_value", "total_freight_value",
    "average_review_score", "repeat_customer",
)
ml_customer_repeat

## Gold Validation

In [ ]:
gold = {
    "customer_analytics": customer_analytics,
    "sales_analytics": sales_analytics,
    "product_analytics": product_analytics,
    "seller_analytics": seller_analytics,
    "delivery_analytics": delivery_analytics,
    "ml_customer_repeat": ml_customer_repeat,
}
gold_keys = {
    "customer_analytics": "customer_unique_id",
    "sales_analytics": "order_id",
    "product_analytics": "product_id",
    "seller_analytics": "seller_id",
    "delivery_analytics": "order_id",
    "ml_customer_repeat": "customer_unique_id",
}

validation_rows = []
for name, frame in gold.items():
    key = gold_keys[name]
    duplicate_key_groups = frame.groupBy(key).count().filter(F.col("count") > 1).count()
    validation_rows.append((name, frame.count(), len(frame.columns), duplicate_key_groups, frame.filter(F.col(key).isNull()).count()))

gold_validation = spark.createDataFrame(
    validation_rows,
    ["dataset", "rows", "columns", "duplicate_key_groups", "null_key_values"],
)
gold_validation.show(truncate=False)

assert all(row.duplicate_key_groups == 0 for row in gold_validation.collect())
assert all(row.null_key_values == 0 for row in gold_validation.collect())

## Write Gold Layer

In [ ]:
gold_output_paths = {
    "customer_analytics": os.path.join(GOLD_PATH, "customer_analytics"),
    "sales_analytics": os.path.join(GOLD_PATH, "sales_analytics"),
    "product_analytics": os.path.join(GOLD_PATH, "product_analytics"),
    "seller_analytics": os.path.join(GOLD_PATH, "seller_analytics"),
    "delivery_analytics": os.path.join(GOLD_PATH, "delivery_analytics"),
    "ml_customer_repeat": os.path.join(GOLD_PATH, "ml_customer_repeat"),
}
for name, frame in gold.items():
    frame.write.mode("overwrite").parquet(gold_output_paths[name])
print(f"Wrote {len(gold)} Gold datasets to {GOLD_PATH}")

## Summary

The Gold layer is split into purpose-specific datasets. Order-level facts are built only after items, payments, and reviews are aggregated to order grain. Customer analytics uses `customer_unique_id`, while order joins use `customer_id`. The repeat-purchase dataset is a starting point for future leakage-safe feature engineering, not a trained model.